<a href="https://colab.research.google.com/github/TrajanDS/SFT_Agent_POC/blob/main/test_gemini_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Uninstall conflicting packages first, if they exist
!pip uninstall -y torch torchvision torchaudio

# Install specific, compatible versions of torch and torchvision for CUDA 12.1
# These versions are known to work well together and with recent transformers
!pip install torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121 -qqq

# Install and upgrade other necessary libraries
!pip install -qqq transformers accelerate bitsandbytes peft trl datasets pyarrow --upgrade

In [ ]:
import google.generativeai as genai
from google.colab import userdata
from tqdm.notebook import tqdm
import pandas as pd
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Define the model name globally
qwen_model_name = "Qwen/Qwen1.5-4B-Chat"

# Load tokenizer globally and configure padding
tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
tokenizer.pad_token = tokenizer.eos_token # Qwen uses EOS as padding token by default
tokenizer.padding_side = "right" # Or "left" depending on preference for inference

In [ ]:
# Mount Google Drive
### (when prompted, approve all permissions and continue)
drive.mount('/content/drive')

# Fetch your API key from Colab Secrets
#GOOGLE_API_KEY = userdata.get('')
genai.configure(api_key='INPUT API KEY HERE')

# Initialize the Gemini model
gemini_model = genai.GenerativeModel('models/gemini-2.5-flash')

# Set output directory (local) path for images and on drive
drive_output_dir = "/content/drive/MyDrive/CFG_complaints/raw_data"

# Read in CFG complaints dataset
comp_df = pd.read_parquet(f"{drive_output_dir}/cfpb_complaints.parquet")

In [ ]:
def classify_product_with_gemini(
    complaint_row,
    product_types = [
        'Checking or savings account',
        'Money transfer, virtual currency, or money service',
        'Vehicle loan or lease',
        'Credit card',
        'Payday loan, title loan, personal loan, or advance loan',
        'Credit reporting or other personal consumer reports',
        'Mortgage',
        'Student loan',
        'Debt collection',
        'Debt or credit management'
    ]
    ):
    prompt = f"""You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list.

Predefined Product Categories: {product_types}

Complaint Details:
Date Received: {complaint_row['date_received']}
Issue: {complaint_row['issue']}
Sub-issue: {complaint_row['sub_issue']}
Company: {complaint_row['company']}
Complaint Narrative: {complaint_row['complaint_what_happened']}

Based on these details, what is the product category for this complaint? Please respond with ONLY the product category string. If you cannot determine a specific product, respond with 'Unknown'."""

    try:
        # Using generate_content for single turn interaction
        response = gemini_model.generate_content(prompt)
        # Accessing response.text assumes the model provides plain text output
        # If the model is fine-tuned to output JSON, you might parse response.text as JSON
        return response.text.strip()
    except Exception as e:
        print(f"Error classifying complaint ID {complaint_row['complaint_id']}: {e}")
        return "Error_Classification"

In [ ]:
tqdm.pandas() # Initialize tqdm for pandas

# Inference gemini model to classify the different product types
gem_inf_df = comp_df.head(5).progress_apply(classify_product_with_gemini, axis=1)

In [ ]:
# Display the value counts of the Gemini-classified products
print("Gemini Classified Product Value Counts:")
display(gem_inf_df['gemini_classified_product'].value_counts())

# Display a comparison of original vs. Gemini classified for a few rows
print("\nComparison of Original vs. Gemini Classified Products (first 10 rows):")
display(gem_inf_df[['product', 'gemini_classified_product', 'complaint_what_happened']].head(10))

# Optionally, calculate accuracy (if original labels are considered ground truth)
correct_classifications = (gem_inf_df['product'] == gem_inf_df['gemini_classified_product']).sum()
total_classifications = len(gem_inf_df)
accuracy = correct_classifications / total_classifications
print(f"\nClassification Accuracy (vs. original 'product' column): {accuracy:.2f}")

## Qwen3.5-4B Model Inference

Now, let's try to classify the complaints using the `Qwen3.5-4B` model. This will involve installing the `transformers` library, loading the model, defining a similar classification function, and then running inference. Please be aware that running a model of this size can be resource-intensive and might be slow or hit memory limits on a standard Colab instance.

In [ ]:
# Load model for initial inference
# This step might take a while and consume significant memory.

# Using bfloat16 for reduced memory usage and potentially faster inference on compatible hardware
# If you encounter CUDA out of memory errors, try using a smaller model or reducing batch size
model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto" # Automatically maps modules to available devices
)
model.eval() # Set model to evaluation mode

In [ ]:
def classify_product_with_qwen(
    complaint_row,
    product_types = [
        'Checking or savings account',
        'Money transfer, virtual currency, or money service',
        'Vehicle loan or lease',
        'Credit card',
        'Payday loan, title loan, personal loan, or advance loan',
        'Credit reporting or other personal consumer reports',
        'Mortgage',
        'Student loan',
        'Debt collection',
        'Debt or credit management'
    ]
    ):
    # Create the prompt for Qwen model, similar to the Gemini prompt
    prompt_text = f"""You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list.

Predefined Product Categories: {product_types}

Complaint Details:
Date Received: {complaint_row['date_received']}
Issue: {complaint_row['issue']}
Sub-issue: {complaint_row['sub_issue']}
Company: {complaint_row['company']}
Complaint Narrative: {complaint_row['complaint_what_happened']}

Based on these details, what is the product category for this complaint? Please respond with ONLY the product category string. If you cannot determine a specific product, respond with 'Unknown'."""

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt_text}
    ]

    # Tokenize the prompt
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False, # We want the string output to then tokenize it manually
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    try:
        # Generate response from Qwen model
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=50, # Limit the output length to get just the category
            do_sample=True, # Use sampling for more varied responses if needed
            temperature=0.1, # Keep temperature low for deterministic classification
            top_p=0.9 # Top-p sampling
        )
        generated_ids = [ output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids) ]
        response_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # Post-process the response to get only the product category
        # The model might output additional text, so we try to extract the category
        predicted_category = "Unknown"
        for p_type in product_types:
            if p_type.lower() in response_text.lower():
                predicted_category = p_type
                break
        return predicted_category
    except Exception as e:
        print(f"Error classifying complaint ID {complaint_row['complaint_id']}: {e}")
        return "Error_Classification"

In [ ]:
print("Running Qwen classification on a single example to demonstrate (due to memory constraints). This may take some time...")
# Clear GPU memory before inference to maximize available VRAM
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Apply the Qwen classification function to a single row to avoid CUDA out of memory
qwen_inf_df = comp_df.head(1).progress_apply(classify_product_with_qwen, axis=1)

# Add the Qwen classifications to a new column in comp_df for comparison
# This will only populate the first row with a classification
comp_df['qwen_classified_product'] = pd.Series(qwen_inf_df, index=comp_df.index)

In [ ]:
# Display the value counts of the Qwen-classified products
print("\nQwen Classified Product Value Counts:")
display(comp_df['qwen_classified_product'].value_counts())

# Display a comparison of original vs. Qwen classified for a few rows
print("\nComparison of Original vs. Qwen Classified Products (first 5 rows):")
display(comp_df[['product', 'qwen_classified_product', 'complaint_what_happened']].head(5))

# Optionally, calculate accuracy (if original labels are considered ground truth)
correct_qwen_classifications = (comp_df['product'] == comp_df['qwen_classified_product']).sum()
total_qwen_classifications = len(comp_df.head(5))
accuracy_qwen = correct_qwen_classifications / total_qwen_classifications
print(f"\nQwen Classification Accuracy (vs. original 'product' column): {accuracy_qwen:.2f}")

## Fine-tuning Qwen3.5-4B with LoRA

To improve the classification performance of Qwen3.5-4B on our specific complaint dataset, we will fine-tune the model using Parameter-Efficient Fine-Tuning (PEFT) with LoRA (Low-Rank Adaptation). This technique allows us to adapt the model to our task by only training a small number of new parameters, significantly reducing computational cost and memory requirements.

### Steps:
1.  **Install necessary libraries**: `bitsandbytes` for 4-bit quantization, `peft` for LoRA, and `trl` for simplified supervised fine-tuning.
2.  **Prepare the dataset**: Format our `comp_df` into a conversational template suitable for instruction fine-tuning, mapping complaint narratives to product types.
3.  **Load the base model**: Load Qwen3.5-4B with 4-bit quantization to save memory.
4.  **Configure LoRA**: Define the LoRA parameters, such as rank and target modules.
5.  **Set up training arguments**: Define hyperparameters for training, like learning rate, batch size, and number of epochs.
6.  **Train the model**: Use `SFTTrainer` from `trl` to perform the fine-tuning.
7.  **Save and merge adapters**: Save the trained LoRA adapters and merge them into the base model.
8.  **Evaluate the fine-tuned model**: Test its classification performance on a separate, unseen subset of the data.

In [ ]:
# Install libraries for PEFT, 4-bit quantization, and ensure latest versions
!pip install -qqq bitsandbytes peft trl transformers datasets pyarrow --upgrade

In [ ]:
# Upgrade pyarrow and datasets to resolve potential import errors
!pip install --upgrade pyarrow datasets -qqq

After running the above cell, please restart the runtime (`Runtime` -> `Restart runtime`) and then proceed to run all cells from the beginning to ensure the updated packages are loaded correctly.

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Filter out rows where 'complaint_what_happened' or 'product' is missing
cleaned_df = comp_df.dropna(subset=['complaint_what_happened', 'product']).copy()

# For demonstration, let's use a smaller subset of the data for fine-tuning
# Fine-tuning on the full dataset might take a very long time.
# We'll use 500 examples for training and 100 for evaluation.
if len(cleaned_df) > 600:
    # Stratified split to ensure product distribution is maintained
    train_df, eval_df = train_test_split(
        cleaned_df.sample(n=600, random_state=42),
        test_size=100,
        random_state=42,
        stratify=cleaned_df.sample(n=600, random_state=42)['product']
    )
else:
    train_df, eval_df = train_test_split(cleaned_df, test_size=0.2, random_state=42)


# Define the formatting function for the chat template
def format_chat_template(row):
    product_types = [
        'Checking or savings account',
        'Money transfer, virtual currency, or money service',
        'Vehicle loan or lease',
        'Credit card',
        'Payday loan, title loan, personal loan, or advance loan',
        'Credit reporting or other personal consumer reports',
        'Mortgage',
        'Student loan',
        'Debt collection',
        'Debt or credit management'
    ]
    messages = [
        {"role": "system", "content": "You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list. Respond with ONLY the product category string."},
        {"role": "user", "content": f"Complaint: {row['complaint_what_happened']}\nPredefined Product Categories: {product_types}"},
        {"role": "assistant", "content": row['product']}
    ]
    # Apply the tokenizer's chat template to format the conversation
    # Note: This requires the tokenizer to be loaded first, but we will define it here for clarity
    # The actual tokenization and formatting will happen within SFTTrainer.
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

# Convert DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df).map(format_chat_template, remove_columns=list(train_df.columns))
eval_dataset = Dataset.from_pandas(eval_df).map(format_chat_template, remove_columns=list(eval_df.columns))

print("Training data sample:")
print(train_dataset[0]['text'])
print("\nEvaluation data sample:")
print(eval_dataset[0]['text'])

In [ ]:
from transformers import BitsAndBytesConfig

# Configure 4-bit quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

# Load the base model with quantization
model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    quantization_config=quantization_config,
    device_map="auto"
)
model.config.use_cache = False
model.config.pretraining_tp = 1 # Recommended for Qwen

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training (important for 4-bit fine-tuning)
model = prepare_model_for_kbit_training(model)

# Define LoRA configuration
lora_config = LoraConfig(
    r=16, # LoRA attention dimension
    lora_alpha=32, # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Target all linear layers in Qwen
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # No bias in LoRA layers
    task_type="CAUSAL_LM", # Causal Language Modeling for instruction tuning
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

print(model.print_trainable_parameters())

In [ ]:
from transformers import TrainingArguments

# Define training arguments
training_args = TrainingArguments(
    output_dir="./qwen_finetuned", # Output directory for checkpoints and logs
    num_train_epochs=3, # Number of training epochs
    per_device_train_batch_size=1, # Batch size per GPU/CPU for training
    gradient_accumulation_steps=4, # Number of updates steps to accumulate gradients
    gradient_checkpointing=True, # Enable gradient checkpointing for memory efficiency
    optim="paged_adamw_8bit", # Optimizer (8-bit AdamW for memory efficiency)
    logging_steps=10, # Log training metrics every N steps
    learning_rate=2e-4, # Learning rate
    bf16=True, # Use bfloat16 if available
    tf32=True, # Use tf32 if available
    max_grad_norm=0.3, # Max gradient norm
    warmup_ratio=0.03, # Warmup ratio for learning rate scheduler
    lr_scheduler_type="constant", # Learning rate scheduler type
    report_to="none", # Disable reporting to W&B or other platforms
    # Evaluation arguments
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

In [ ]:
from trl import SFTTrainer

# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=512, # Maximum sequence length for inputs
    dataset_text_field="text", # Field in dataset containing the formatted text
    packing=True, # Pack multiple examples into one sequence for efficiency
)

# Start training
print("\nStarting Qwen fine-tuning...")
trainer.train()
print("\nFine-tuning complete!")

In [ ]:
# Save the fine-tuned adapter weights
output_dir = "./qwen_finetuned_adapters"
trainer.model.save_pretrained(output_dir)

# To use the fine-tuned model for inference, we need to load the base model
# and then merge the adapters. If you want to share, save the merged model.

# Clear VRAM if needed
del model
import gc
gc.collect()
torch.cuda.empty_cache()

# Reload the base model and tokenizer (without quantization for merging)
base_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    return_dict=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load the PEFT model and merge
from peft import PeftModel
finetuned_model = PeftModel.from_pretrained(base_model, output_dir)
finetuned_model = finetuned_model.merge_and_unload()
finetuned_model.eval()

print("Fine-tuned model loaded and adapters merged successfully.")

In [ ]:
# Update the classification function to use the fine-tuned model
def classify_product_with_finetuned_qwen(
    complaint_row,
    model,
    tokenizer,
    product_types = [
        'Checking or savings account',
        'Money transfer, virtual currency, or money service',
        'Vehicle loan or lease',
        'Credit card',
        'Payday loan, title loan, personal loan, or advance loan',
        'Credit reporting or other personal consumer reports',
        'Mortgage',
        'Student loan',
        'Debt collection',
        'Debt or credit management'
    ]
    ):
    messages = [
        {"role": "system", "content": "You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list. Respond with ONLY the product category string."},
        {"role": "user", "content": f"Complaint: {complaint_row['complaint_what_happened']}\nPredefined Product Categories: {product_types}"}
    ]

    # Tokenize the prompt
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    try:
        # Generate response from Qwen model
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=50, # Limit the output length
            do_sample=False, # Use greedy decoding for deterministic classification
            temperature=0.0, # Set temperature to 0 for deterministic output
        )
        # Decode only the newly generated tokens
        input_len = model_inputs.input_ids.shape[1]
        response_text = tokenizer.decode(generated_ids[0, input_len:], skip_special_tokens=True).strip()

        # Post-process the response to get only the product category
        predicted_category = "Unknown"
        for p_type in product_types:
            if p_type.lower() in response_text.lower():
                predicted_category = p_type
                break
        # Fallback to the direct response if no predefined category is found and response is short
        if predicted_category == "Unknown" and len(response_text.split()) < 5: # Assuming category is short
             predicted_category = response_text # Take the model's direct output

        return predicted_category
    except Exception as e:
        print(f"Error classifying complaint ID {complaint_row['complaint_id']}: {e}")
        return "Error_Classification"

print("Running fine-tuned Qwen classification on the evaluation subset (first 10 rows)...")
# Apply the fine-tuned Qwen classification function to the evaluation set
finetuned_qwen_inf_df = eval_df.head(10).progress_apply(
    lambda row: classify_product_with_finetuned_qwen(row, finetuned_model, tokenizer),
    axis=1
)

# Add the Qwen classifications to a new column in eval_df for comparison
eval_df = eval_df.head(10).copy() # Use the subset to match inference results
eval_df['finetuned_qwen_classified_product'] = finetuned_qwen_inf_df

# Display the value counts of the fine-tuned Qwen-classified products
print("\nFine-tuned Qwen Classified Product Value Counts:")
display(eval_df['finetuned_qwen_classified_product'].value_counts())

# Display a comparison of original vs. fine-tuned Qwen classified for a few rows
print("\nComparison of Original vs. Fine-tuned Qwen Classified Products (first 10 rows):")
display(eval_df[['product', 'finetuned_qwen_classified_product', 'complaint_what_happened']])

# Calculate accuracy on the evaluation subset
correct_finetuned_qwen_classifications = (eval_df['product'] == eval_df['finetuned_qwen_classified_product']).sum()
total_finetuned_qwen_classifications = len(eval_df)
accuracy_finetuned_qwen = correct_finetuned_qwen_classifications / total_finetuned_qwen_classifications
print(f"\nFine-tuned Qwen Classification Accuracy (vs. original 'product' column): {accuracy_finetuned_qwen:.2f}")